In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df["Partner"] = df["Partner"].fillna(df["Partner"].mode()[0])
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].median())
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [5]:
df["gender"] = df["gender"].replace({ "Male" : 1, "Female" : 0})
df["Partner"] = df["Partner"].replace({ "Yes" : 1, "No" : 0})
df["Dependents"] = df["Dependents"].replace({ "Yes" : 1, "No" : 0})
df["PhoneService"] = df["PhoneService"].replace({ "Yes" : 1, "No" : 0})
df["MultipleLines"] = df["MultipleLines"].replace({ "Yes" : 1, "No" : 0, "No phone service" : 0 })
df["OnlineSecurity"] = df["OnlineSecurity"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["OnlineBackup"] = df["OnlineBackup"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["DeviceProtection"] = df["DeviceProtection"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["TechSupport"] = df["TechSupport"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["StreamingTV"] = df["StreamingTV"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["StreamingMovies"] = df["StreamingMovies"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["PaperlessBilling"] = df["PaperlessBilling"].replace({ "Yes" : 1, "No" : 0 })
df["StreamingTV"] = df["StreamingTV"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["Churn"] = df["Churn"].replace({"Yes": 1, "No": 0})
df = pd.get_dummies(df, columns=["InternetService", "Contract", "PaymentMethod"])

C:\Users\darag\AppData\Local\Temp\ipykernel_22416\3653414510.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["gender"] = df["gender"].replace({ "Male" : 1, "Female" : 0})
C:\Users\darag\AppData\Local\Temp\ipykernel_22416\3653414510.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Partner"] = df["Partner"].replace({ "Yes" : 1, "No" : 0})
C:\Users\darag\AppData\Local\Temp\ipykernel_22416\3653414510.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain th

In [6]:
X = df.drop(columns=["customerID", "Churn"])
Y = df["Churn"]

In [7]:
X_temp, X_test, Y_temp, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42)

In [8]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X_temp, Y_temp, test_size=0.2, random_state=42)

In [9]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

Step 1: Deliberately overfit a model (e.g. a very deep decision tree) and confirm the large train-vs
validation gap.

In [15]:
overfit_model = DecisionTreeClassifier(random_state=42, class_weight='balanced')
overfit_model.fit(X_train_scaled, Y_train)

f1_train = f1_score(Y_train, overfit_model.predict(X_train_scaled))
f1_val = f1_score(Y_val, overfit_model.predict(X_val_scaled))
f1_gap = f1_train - f1_val

print(f"f1 train = {f1_train:.3f}")
print(f"f1 val = {f1_val:.3f}")
print(f"the gap  = {f1_gap:.3f}")

f1 train = 0.997
f1 val = 0.500
the gap  = 0.497


here we can see that there is very large gap between train(0.997) and validation(0.500) data.
for this task we use Decision Tree Classifier model without limited depth, so the model keep tracking the data until it memorize it.
the gap bwtween the data = 0.497, meaning that we have an overfitting.

Step 2: Deliberately underfit a model (e.g. an overly simple one) and confirm both scores are low.

In [16]:
underfit_model = DecisionTreeClassifier(max_depth= 1, random_state=42, class_weight='balanced')
underfit_model.fit(X_train_scaled, Y_train)

f1_train = f1_score(Y_train, underfit_model.predict(X_train_scaled))
f1_val = f1_score(Y_val, underfit_model.predict(X_val_scaled))
f1_gap = f1_train - f1_val

print(f"f1 train = {f1_train:.3f}")
print(f"f1 val = {f1_val:.3f}")
print(f"the gap  = {f1_gap:.3f}")

f1 train = 0.575
f1 val = 0.573
the gap  = 0.002


here we made a very simple model which depth is 1, the model is too simple so it's give us a low predictions in both training(0.575) and val(0.573) data.
the gap between them is too small(0.002) 'Cause the model doesn't do well on both training and validation data. this is underfitting model.

Step 3: Apply regularization (or reduce complexity) to the overfit model and show the gap shrink.

first I want to show how to solve this problem with DecisionTreeClassifier, we can't use L1 & L2 on Decision Tree so to solve it we will give the model restrictions.

In [17]:
regularized_model = DecisionTreeClassifier(
    max_depth= 4,
    min_samples_leaf = 20,
    min_samples_split=40,
    random_state=42,
    class_weight='balanced')
regularized_model.fit(X_train_scaled, Y_train)

for name,model in [("overfit", overfit_model), ("regularized", regularized_model)]:
    train_f1 = f1_score(Y_train, model.predict(X_train_scaled))
    val_f1 = f1_score(Y_val, model.predict(X_val_scaled))
    print(f"{name} - train: {train_f1:.4f}, validation: {val_f1:.4f}, gap = {(train_f1 - val_f1):.4f}")

overfit - train: 0.9975, validation: 0.5000, gap = 0.4975
regularized - train: 0.6329, validation: 0.6045, gap = 0.0284


note that the train f1 dropped from 0.9975 to 0.6329 after regularization this is expected and good. it meana the model stopped memorizing the training and started generalizing better. the gap is now small (0.0284), which is much closer to a good fit, though not perfectly zero.

Step 4: Document each diagnosis and fix with the score evidence in Markdown.

model1 - underfit (max_depth=1)/ train f1 = 0.575/ val f1 = 0.573/ gap = 0.002/ underfitting - model too simple.

model2 - overfit (no limit)/ train f1 = 0.9975/ val f1 = 0.5000/ gap = 0.4975/ overfitting - model memorized training.

model3 - regularized (max_depth=4)/ train f1 = 0.6329/ val f1 = 0.6045/ gap = 0.0284/ good fit - close balance.